# 04 – Datavoorbereiding: van ruwe data naar de modeltabel over alle jaren (zonder grafieken)

| | |
|---|---|
| **Auteur(s)** | Kobe Vandenberk |
| **Wat** | Alle cleaning- en voorbereidingsstappen uit `02_eda_cleaning` (2024), `02b_eda_all_years` en `03_hypotheses`, achter elkaar en zonder grafieken. Samen met `01_download` is dit de volledige weg van de ruwe zip's (2013–2026) naar de tabel waarop het definitieve model traint. |
| **Input** | `data/station_hour/*.parquet`, `data/stations.parquet`, `data/weather.parquet` (uit `01_download`) |
| **Output** | `data/model_table.parquet`, `data/zones.json`, `data/station_zones.parquet` |
| **GenAI** | Code en tekst opgesteld met hulp van Claude; keuzes en resultaten nagekeken door Kobe. |

## Stappen (en waar ze onderbouwd worden)
| Stap | Wat | Waarom (zie) |
|---|---|---|
| 1 | Alle formaten naar één schema; per maand één versie van de data (al in 01) | 02b §1–3 |
| 2 | Alleen vertrekken met starttijd, startstation en coördinaten in NYC (al in 01) | 02 §2 |
| 3 | Station-id's normaliseren `3423.1` → `3423.10` (al in 01) | 02 §3 |
| 4 | Niet-publieke stations en Jersey City/Hoboken weglaten | 02 §5 |
| 5 | 30 zones (KMeans) op het netwerk van de **laatste 12 maanden**; elk station uit elk jaar krijgt de zone van zijn locatie | 02 §8, 02b §5 en §7 |
| 6 | Rooster zone × uur over alle jaren, uren zonder vertrek = 0 | 02 §10 |
| 7 | Kenmerk **actieve stations** per zone en maand; zone-uren zonder enig actief station weglaten | 02b §7 |
| 8 | Dagen met een gat in de data weglaten (`cb.EXCLUDE_DAYS`) | 02b §6 |
| 9 | Weer per uur erbij (Open-Meteo), inclusief **sneeuwhoogte** | 03, 09 |
| 10 | Split: de **laatste 12 maanden** = test | hieronder |

**Waarom een andere split dan in 2024?** Daar testten we op de laatste week van elke maand, binnen één jaar. Met 13 jaar data meten we zoals het model gebruikt wordt: getraind op het verleden, getest op de meest recente 12 maanden (elk seizoen één keer). Voor het tunen gebruiken we in `05e` **backtesting**: trainen op alle jaren vóór een validatiejaar en valideren op dat jaar.

In [1]:
import json

import pandas as pd

import citibike as cb

stations = pd.read_parquet(cb.DATA_DIR / "stations.parquet")
weather = pd.read_parquet(cb.DATA_DIR / "weather.parquet")
print(f"{len(stations):,} station-id's, {len(weather):,} weer-uren, uit te sluiten dagen: {len(cb.EXCLUDE_DAYS)}")

3,824 station-id's, 116,880 weer-uren, uit te sluiten dagen: 12


## Stappen 4–10

In [2]:
table, zones, public_stations = cb.prepare_full_table(cb.DATA_DIR / "station_hour", stations, weather,
                                                      exclude_days=cb.EXCLUDE_DAYS)
dropped = cb.clean_stations(stations).query("not keep")
print(f"Weggelaten stations: {len(dropped)} ({dropped['trips'].sum():,} vertrekken)")
print(f"Modeltabel: {len(table):,} zone-uren van {table['time'].min()} tot {table['time'].max()}")
table.head()

Weggelaten stations: 110 (18,813 vertrekken)
Modeltabel: 2,818,992 zone-uren van 2013-06-01 00:00:00 tot 2026-09-30 23:00:00


,zone,time,trips,members,electric,active_stations,temperature_c,precipitation_mm,wind_kmh,snowfall_cm,humidity_pct,snow_depth_cm,split
0,0,2013-06-01 00:00:00,24,21,0,32,22.7,0.0,11.2,0.0,65,0.0,train
1,0,2013-06-01 01:00:00,10,9,0,32,21.9,0.0,10.5,0.0,70,0.0,train
2,0,2013-06-01 02:00:00,7,4,0,32,21.2,0.0,10.5,0.0,75,0.0,train
3,0,2013-06-01 03:00:00,8,3,0,32,20.6,0.0,9.7,0.0,78,0.0,train
4,0,2013-06-01 04:00:00,2,2,0,32,20.1,0.0,9.7,0.0,83,0.0,train


## Controles
1. Geen dubbele (zone, uur)-rijen en geen ontbrekende waarden.
2. **Er gaan geen vertrekken verloren**, behalve die op de uitgesloten dagen: de som van `trips` is gelijk aan alle vertrekken van de publieke stations op de overige dagen.
3. Elke rij heeft minstens één actief station.
4. De testset beslaat precies de laatste 12 maanden.

In [3]:
assert not table.duplicated(["zone", "time"]).any()
assert table.notna().all().all(), table.isna().sum()
assert (table["active_stations"] > 0).all()

kept = set(public_stations["start_station_id"])
excluded = pd.to_datetime(list(cb.EXCLUDE_DAYS))
expected = 0
for path in sorted((cb.DATA_DIR / "station_hour").glob("*.parquet")):
    sh = pd.read_parquet(path, columns=["start_station_id", "time", "trips"])
    expected += sh.loc[sh["start_station_id"].isin(kept) & ~sh["time"].dt.normalize().isin(excluded), "trips"].sum()
assert table["trips"].sum() == expected, (table["trips"].sum(), expected)

test = table[table["split"] == "test"]
print(f"OK: {len(table):,} rijen, {table['trips'].sum():,} vertrekken; test van {test['time'].min()} tot "
      f"{test['time'].max()} ({len(test) / len(table):.1%} van de rijen)")

OK: 2,818,992 rijen, 328,688,926 vertrekken; test van 2025-10-01 00:00:00 tot 2026-09-30 23:00:00 (9.3% van de rijen)


## Opslaan
- `model_table.parquet`: de tabel voor het definitieve model (en voor `train.py`). Dit bestand staat in Git, zodat iedereen het model kan hertrainen zonder 32 GB te downloaden.
- `zones.json`: id, naam, centrum, grootte en het **huidige aantal actieve stations** van elke zone (de API gebruikt dat laatste als kenmerk bij voorspellingen voor de toekomst).
- `station_zones.parquet`: welk station (uit welk jaar) in welke zone ligt.

In [4]:
columns = ["zone", "time", "trips", "members", "electric", "active_stations", "temperature_c", "precipitation_mm",
           "wind_kmh", "snowfall_cm", "snow_depth_cm", "humidity_pct", "split"]
table = table[columns].astype({"zone": "int16", "trips": "int32", "members": "int32", "electric": "int32",
                               "active_stations": "int16"})
table.to_parquet(cb.DATA_DIR / "model_table.parquet", index=False)
zones_records = zones.assign(lat=zones["lat"].round(5), lng=zones["lng"].round(5)).to_dict("records")
(cb.DATA_DIR / "zones.json").write_text(json.dumps(zones_records, indent=2, default=int), encoding="utf-8")
public_stations.to_parquet(cb.DATA_DIR / "station_zones.parquet", index=False)
print(f"model_table.parquet: {(cb.DATA_DIR / 'model_table.parquet').stat().st_size / 1e6:.1f} MB")
pd.DataFrame(zones_records)[["zone", "name", "n_stations", "active_stations", "trips"]].head(10)

model_table.parquet: 27.8 MB


,zone,name,n_stations,active_stations,trips
0,0,rond Broadway & E 14 St,55,54,3171045
1,1,rond W 21 St & 6 Ave,48,44,3117937
2,2,rond Norfolk St & Broome St,52,52,2691235
3,3,rond Park Ave & E 42 St,47,47,2687753
4,4,rond W 31 St & 7 Ave,43,42,2686707
5,5,rond Metropolitan Ave & Bedford Ave,77,75,2497805
6,6,rond Broadway & W 58 St,42,40,2404518
7,7,rond Carmine St & 6 Ave,46,42,2379871
8,8,rond Schermerhorn St & Hoyt St,79,79,2069796
9,9,rond West St & Liberty St,56,50,1855619
